# Clase 055 — Feature engineering avanzado: target encoding + MICE

Dataset sintético de clasificación con 3 categóricas de alta cardinalidad (1000 niveles) + missing values.
Comparamos one-hot vs target encoding, y SimpleImputer vs MICE.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge

rng = np.random.default_rng(42)
np.random.seed(42)

## 1. Dataset sintético

1000 niveles por columna, target con señal en las categorías (cada nivel tiene una probabilidad latente).

In [ ]:
n_samples, n_levels = 5000, 1000

# Probabilidades latentes por nivel (lo que aprenderá target encoding)
latent_a = rng.beta(2, 2, n_levels)
latent_b = rng.beta(2, 2, n_levels)
latent_c = rng.beta(2, 2, n_levels)

cat_a = rng.integers(0, n_levels, n_samples)
cat_b = rng.integers(0, n_levels, n_samples)
cat_c = rng.integers(0, n_levels, n_samples)
num_1 = rng.normal(0, 1, n_samples)
num_2 = rng.normal(0, 1, n_samples)

logit = (latent_a[cat_a] - 0.5) + (latent_b[cat_b] - 0.5) + 0.3 * num_1 + 0.2 * num_2
p = 1 / (1 + np.exp(-3 * logit))
y = (rng.uniform(0, 1, n_samples) < p).astype(int)

df = pd.DataFrame({'cat_a': cat_a, 'cat_b': cat_b, 'cat_c': cat_c, 'num_1': num_1, 'num_2': num_2})
print('shape', df.shape, 'pos rate', y.mean().round(3))

## 🧠 Intuición previa

Dos ideas que parecen complicadas y en el fondo son simples. **Target encoding** reemplaza cada categoría por el promedio del target en esa categoría, pero *suavizado*: si una categoría aparece poquísimas veces, se la mezcla con el promedio global para no confiar de más en un dato flaco (como fiarse de un producto con 2 reseñas vs uno con 2000). **MICE** (imputación iterativa) rellena los faltantes prediciendo cada columna con NaN a partir de las demás, y repite el ciclo varias veces hasta que las estimaciones se estabilizan, en lugar de tapar todo con la media.

## 2. Target encoding manual con smoothing bayesiano

$\text{enc}(c) = \frac{n_c \cdot \bar{y}_c + k \cdot \bar{y}_{\text{global}}}{n_c + k}$

In [ ]:
def target_encoding(x_train, y_train, x_test, smoothing=10):
    """Target encoding con smoothing bayesiano. Fitted en train, aplicado a ambos."""
    global_mean = y_train.mean()
    df_t = pd.DataFrame({'x': x_train, 'y': y_train})
    agg = df_t.groupby('x')['y'].agg(['mean', 'count'])
    enc = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
    return (pd.Series(x_train).map(enc).fillna(global_mean).values,
            pd.Series(x_test).map(enc).fillna(global_mean).values)

def loo_target_encoding(x, y, smoothing=10):
    """Leave-one-out target encoding (sin leakage)."""
    global_mean = y.mean()
    df_t = pd.DataFrame({'x': x, 'y': y})
    agg = df_t.groupby('x')['y'].agg(['sum', 'count'])
    s = pd.Series(x).map(agg['sum']).values
    c = pd.Series(x).map(agg['count']).values
    return ((s - y) + smoothing * global_mean) / ((c - 1) + smoothing)

X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.3, random_state=42, stratify=y)

## 3. One-hot vs target encoding (LogReg + GBM)

In [ ]:
# Target encoding
X_train_te = X_train[['num_1', 'num_2']].copy()
X_test_te = X_test[['num_1', 'num_2']].copy()
for col in ['cat_a', 'cat_b', 'cat_c']:
    tr, te = target_encoding(X_train[col].values, y_train, X_test[col].values, smoothing=10)
    X_train_te[col + '_te'] = tr
    X_test_te[col + '_te'] = te

# One-hot (sparse para no reventar RAM con 3000 columnas)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_oh = ohe.fit_transform(X_train[['cat_a', 'cat_b', 'cat_c']])
X_test_oh = ohe.transform(X_test[['cat_a', 'cat_b', 'cat_c']])
print(f'one-hot shape: {X_train_oh.shape} ({X_train_oh.nnz} nnz)')
print(f'target enc shape: {X_train_te.shape}')

In [ ]:
results = {}

# LogReg + one-hot
lr = LogisticRegression(max_iter=300, random_state=42)
lr.fit(X_train_oh, y_train)
results['LogReg + OneHot'] = roc_auc_score(y_test, lr.predict_proba(X_test_oh)[:, 1])

# LogReg + target encoding
lr2 = LogisticRegression(max_iter=300, random_state=42)
lr2.fit(X_train_te, y_train)
results['LogReg + TargetEnc'] = roc_auc_score(y_test, lr2.predict_proba(X_test_te)[:, 1])

# GBM + target encoding
gbm = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
gbm.fit(X_train_te, y_train)
results['GBM + TargetEnc'] = roc_auc_score(y_test, gbm.predict_proba(X_test_te)[:, 1])

print(pd.Series(results).round(4).to_string())

## 4. Missing values: MCAR vs MAR

- **MCAR** (missing completely at random): la probabilidad de NaN no depende de nada.
- **MAR** (missing at random): la probabilidad de NaN depende de otra feature observada.

In [ ]:
X_num = pd.DataFrame({
    'a': rng.normal(0, 1, 2000),
    'b': rng.normal(0, 1, 2000),
    'c': rng.normal(0, 1, 2000),
})
X_num['b'] = 0.7 * X_num['a'] + 0.3 * X_num['b']  # a y b correlacionadas
X_num['c'] = 0.5 * X_num['a'] + 0.5 * X_num['c']

# MCAR: 20% random
X_mcar = X_num.copy()
mask_mcar = rng.uniform(0, 1, X_num.shape) < 0.2
X_mcar[mask_mcar] = np.nan

# MAR: NaN en 'b' depende de 'a' (cuando a alto, b se pierde)
X_mar = X_num.copy()
X_mar.loc[X_mar['a'] > 0.5, 'b'] = np.nan

print('MCAR NaN ratio:', X_mcar.isna().mean().round(3).to_dict())
print('MAR  NaN ratio:', X_mar.isna().mean().round(3).to_dict())

## 5. SimpleImputer vs IterativeImputer (MICE)

Medimos el error de reconstrucción: imputado vs valor verdadero.

In [ ]:
def eval_imputer(X_true, X_with_nan, imputer):
    mask = X_with_nan.isna().values
    X_imp = imputer.fit_transform(X_with_nan)
    err = np.abs(X_imp[mask] - X_true.values[mask]).mean()
    return err

results = []
for name, X_nan in [('MCAR', X_mcar), ('MAR', X_mar)]:
    simple_mean = eval_imputer(X_num, X_nan, SimpleImputer(strategy='mean'))
    mice = eval_imputer(X_num, X_nan,
                        IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42))
    results.append({'tipo': name, 'SimpleImputer (mean)': simple_mean, 'MICE (BayesianRidge)': mice})

print(pd.DataFrame(results).round(4).to_string(index=False))
print('\nMICE gana fuerte en MAR porque usa la correlación a↔b para predecir b cuando falta.')

## 6. Sesgo de imputación simple

SimpleImputer comprime la varianza (todos los NaN → mismo valor); MICE preserva la distribución.

In [ ]:
X_imp_simple = pd.DataFrame(
    SimpleImputer(strategy='mean').fit_transform(X_mar), columns=X_num.columns)
X_imp_mice = pd.DataFrame(
    IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42).fit_transform(X_mar),
    columns=X_num.columns)

print('std verdadero:', X_num.std().round(3).to_dict())
print('std SimpleImp:', X_imp_simple.std().round(3).to_dict())
print('std MICE     :', X_imp_mice.std().round(3).to_dict())
print('\ncorr(a,b) verdadero:', X_num.corr().loc["a","b"].round(3))
print('corr(a,b) SimpleImp:', X_imp_simple.corr().loc["a","b"].round(3))
print('corr(a,b) MICE     :', X_imp_mice.corr().loc["a","b"].round(3))

## Ejercicios

1. Implementá CatBoost-style ordered target encoding y compará con smoothing.
2. Probá `KNNImputer` y agregálo a la tabla.
3. Aplicá MICE con `RandomForestRegressor` como estimador.

## Conclusiones

- Target encoding con smoothing gana fuerte cuando hay alta cardinalidad.
- LOO o CV evita el leakage clásico (cada sample no ve su propio target).
- MICE supera a SimpleImputer especialmente en MAR — explota correlaciones.
- SimpleImputer aplasta la varianza y subestima correlaciones.

## ✅ Soluciones de los ejercicios

Resolvemos los 5 ejercicios del README con datasets sintéticos offline. Para el encoding con CV usamos `sklearn.preprocessing.TargetEncoder`; el 'CatBoost-style' lo implementamos como leave-one-out manual (sin dependencias externas).

**Ej. 1 — Target encoding leak.** Calcular la media del target por categoría sobre **todo** el dataset (incluida la fila misma / el test) infla la métrica. Comparado con ajustar el encoder solo en train.

In [ ]:

import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
n, levels = 4000, 1500                       # ~2-3 filas por categoria: cardinalidad alta
cat = rng.integers(0, levels, n)
latent = rng.beta(2, 2, levels)
yv = (rng.uniform(0, 1, n) < latent[cat]).astype(int)
dfx = pd.DataFrame({"cat": cat.astype(str)})
Xtr, Xte, ytr, yte = train_test_split(dfx, yv, test_size=0.3, random_state=42, stratify=yv)
gm = ytr.mean()

# CON LEAK: encoding con la media por categoria calculada sobre TODO (train+test)
enc_leak = pd.DataFrame({"cat": dfx["cat"].values, "y": yv}).groupby("cat")["y"].mean()
Xtr_l = Xtr["cat"].map(enc_leak).fillna(gm).values.reshape(-1, 1)
Xte_l = Xte["cat"].map(enc_leak).fillna(gm).values.reshape(-1, 1)
auc_leak = roc_auc_score(yte, LogisticRegression().fit(Xtr_l, ytr).predict_proba(Xte_l)[:, 1])

# SIN LEAK: encoding ajustado SOLO en train
enc_ok = pd.DataFrame({"cat": Xtr["cat"].values, "y": ytr}).groupby("cat")["y"].mean()
Xtr_o = Xtr["cat"].map(enc_ok).fillna(gm).values.reshape(-1, 1)
Xte_o = Xte["cat"].map(enc_ok).fillna(gm).values.reshape(-1, 1)
auc_ok = roc_auc_score(yte, LogisticRegression().fit(Xtr_o, ytr).predict_proba(Xte_o)[:, 1])

print(f"AUC con leak (encoder sobre todo): {auc_leak:.3f}  <- optimista, IMPOSIBLE en produccion")
print(f"AUC sin leak (encoder solo train): {auc_ok:.3f}  <- realista")
assert auc_leak > auc_ok + 0.05, "el leak debe inflar la metrica de forma notoria"

**Ej. 2 — Target encoding con CV (sin leak).** `TargetEncoder(smooth=10, cv=5)` dentro de un pipeline codifica cada fold con datos de los otros folds: encoding potente sin filtrar el target.

In [ ]:

import numpy as np, pandas as pd
from sklearn.preprocessing import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(1)
n, levels = 4000, 300
cat = rng.integers(0, levels, n)
latent = rng.beta(2, 2, levels)
yv = (rng.uniform(0, 1, n) < latent[cat]).astype(int)
Xz = pd.DataFrame({"cat": cat.astype(str)})

pipe = Pipeline([("te", TargetEncoder(smooth=10.0, cv=5, random_state=42)),
                 ("lr", LogisticRegression(max_iter=500))])
auc = cross_val_score(pipe, Xz, yv, cv=5, scoring="roc_auc")
print(f"AUC CV con TargetEncoder(cv=5): {auc.mean():.3f} +/- {auc.std():.3f}  (honesto, sin leak)")
assert 0.5 < auc.mean() < 0.99

**Ej. 3 — Alternativa estilo CatBoost (ordered / leave-one-out).** El LOO encoding excluye la fila propia al calcular la media de su categoría; evita el leak sin necesidad de CV externo. Lo comparamos contra el smoothing simple.

In [ ]:

import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(2)
n, levels = 4000, 300
cat = rng.integers(0, levels, n)
latent = rng.beta(2, 2, levels)
yv = (rng.uniform(0, 1, n) < latent[cat]).astype(int)
Xtr, Xte, ytr, yte = train_test_split(np.arange(n), yv, test_size=0.3, random_state=42, stratify=yv)
ctr, cte = cat[Xtr], cat[Xte]
gm = ytr.mean()

# LOO en train (cada fila NO cuenta su propio y); test usa la media plena de train
agg = pd.DataFrame({"c": ctr, "y": ytr}).groupby("c")["y"].agg(["sum", "count"])
s = pd.Series(ctr).map(agg["sum"]).values
cnt = pd.Series(ctr).map(agg["count"]).values
loo_tr = ((s - ytr) / np.maximum(cnt - 1, 1))
loo_tr = np.where(cnt > 1, loo_tr, gm).reshape(-1, 1)
mean_map = (agg["sum"] / agg["count"])
loo_te = pd.Series(cte).map(mean_map).fillna(gm).values.reshape(-1, 1)
auc_loo = roc_auc_score(yte, LogisticRegression().fit(loo_tr, ytr).predict_proba(loo_te)[:, 1])

# smoothing simple (k=10)
enc = (agg["count"] * mean_map + 10 * gm) / (agg["count"] + 10)
sm_tr = pd.Series(ctr).map(enc).fillna(gm).values.reshape(-1, 1)
sm_te = pd.Series(cte).map(enc).fillna(gm).values.reshape(-1, 1)
auc_sm = roc_auc_score(yte, LogisticRegression().fit(sm_tr, ytr).predict_proba(sm_te)[:, 1])
print(f"AUC LOO (estilo CatBoost): {auc_loo:.3f}")
print(f"AUC smoothing (k=10)     : {auc_sm:.3f}")
print("ambos evitan el leak; en cardinalidad alta suelen quedar parejos")
assert min(auc_loo, auc_sm) > 0.5

**Ej. 4 — `KNNImputer` vs `SimpleImputer`.** Con columnas correlacionadas, KNN aprovecha los vecinos para reconstruir mejor que rellenar con la media.

In [ ]:

import numpy as np, pandas as pd
from sklearn.impute import KNNImputer, SimpleImputer

rng = np.random.default_rng(3)
base = pd.DataFrame({"a": rng.normal(0, 1, 2000)})
base["b"] = 0.8 * base["a"] + 0.2 * rng.normal(0, 1, 2000)   # b correlaciona con a
base["c"] = 0.5 * base["a"] + 0.5 * rng.normal(0, 1, 2000)
Xnan = base.copy()
mask = rng.uniform(0, 1, base.shape) < 0.2
Xnan[mask] = np.nan

def recon_err(imp):
    Ximp = imp.fit_transform(Xnan)
    return np.abs(Ximp[mask] - base.values[mask]).mean()

e_simple = recon_err(SimpleImputer(strategy="mean"))
e_knn = recon_err(KNNImputer(n_neighbors=5))
print(f"error reconstruccion SimpleImputer(mean): {e_simple:.4f}")
print(f"error reconstruccion KNNImputer(k=5)    : {e_knn:.4f}")
assert e_knn < e_simple, "KNN aprovecha la correlacion y reconstruye mejor"

**Ej. 5 — MICE (`IterativeImputer` + `BayesianRidge`).** MICE modela cada columna a partir de las otras; gana sobre la media, sobre todo cuando el faltante es MAR (depende de otra variable observada).

In [ ]:

import numpy as np, pandas as pd
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge

rng = np.random.default_rng(4)
base = pd.DataFrame({"a": rng.normal(0, 1, 2000)})
base["b"] = 0.7 * base["a"] + 0.3 * rng.normal(0, 1, 2000)
base["c"] = 0.5 * base["a"] + 0.5 * rng.normal(0, 1, 2000)
Xmar = base.copy()
Xmar.loc[Xmar["a"] > 0.5, "b"] = np.nan          # MAR: b se pierde cuando a es alto
mask = Xmar.isna().values

def recon_err(imp):
    Ximp = imp.fit_transform(Xmar)
    return np.abs(Ximp[mask] - base.values[mask]).mean()

e_simple = recon_err(SimpleImputer(strategy="mean"))
e_mice = recon_err(IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42))
print(f"MAR  SimpleImputer(mean): {e_simple:.4f}")
print(f"MAR  MICE (BayesianRidge): {e_mice:.4f}")
assert e_mice < e_simple, "MICE explota la correlacion a<->b y reconstruye mejor en MAR"